# Federated Learning for AML: Seeing "Before vs After" with Your Own Eyes

This hands-on notebook walks through a complete Federated Learning (FL)
workflow for AML (Anti-Money Laundering) fraud detection, built on the
[Flower](https://flower.ai) framework and the PaySim synthetic transaction
dataset. The goal is to make FL's value **visible**: heatmaps and charts that
contrast what each bank can do alone with what all banks achieve together.

## Storyline

| Stage | What happens | Visualization |
|---|---|---|
| Before | 10 banks each train on their own data only ("every bank for itself") | Cross-evaluation heatmap: banks with few fraud labels fail badly |
| Principle | Compute FedAvg by hand, with no infrastructure | - |
| FL run | Real distributed processes (1 SuperLink + 10 SuperNodes) run FedAvg over gRPC | Round-by-round progress monitoring |
| After | Compare the federated model against every solo model, under identical data constraints | Before/After heatmap, confusion matrices, convergence chart |

The central question: **"My bank has only 18 confirmed fraud cases. Can I get
a strong detection model without ever sharing my raw data?"** By the end of
this notebook you will see FL's answer in numbers and pictures.

## Prerequisites

> ### RUN IN TERMINAL (one-time setup)
> These commands create the Python environment, register the Jupyter kernel,
> and start Jupyter. They must be executed in a terminal on this machine:
>
> ```bash
> cd <repository-root>
>
> python3 -m venv --system-site-packages .venv
> .venv/bin/pip install "flwr==1.36.0" pyarrow scikit-learn pandas torch matplotlib jupyter ipykernel
> .venv/bin/python -m ipykernel install --user --name fl-aml --display-name "Python (fl-AML)"
>
> .venv/bin/python -m jupyterlab --no-browser --ip 0.0.0.0
> ```

Open this notebook from the repository root and select the
**Python (fl-AML)** kernel. Total time: roughly 30-40 minutes, including a
10-minute FL run (with live monitoring while you wait).

---
## 1. Environment Check

In [ ]:
import sys, json, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT = Path.cwd()
assert (PROJECT / "pyproject.toml").exists(), (
    "Run this notebook from the repository root "
    "(start Jupyter in the directory that contains pyproject.toml).")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

import flwr, sklearn, pandas as pd, numpy as np, torch
print("project:", PROJECT)
print("python :", sys.executable)
print("flwr:", flwr.__version__, "| sklearn:", sklearn.__version__,
      "| torch:", torch.__version__)

---
## 2. Getting and Verifying the Data

PaySim is a synthetic mobile-money transaction dataset (original:
Kaggle `ealaxi/paysim1`). The cell below downloads it from a public
HuggingFace mirror (no Kaggle credentials needed) and verifies its integrity
with hard asserts. Expected: **6,362,620 rows / 8,213 frauds (0.129%)** - an
extreme class imbalance.

The download is skipped if the file already exists, so the cell is safe to
re-run (first run takes about 5-8 minutes).

In [ ]:
r = subprocess.run(
    [sys.executable, str(PROJECT / "scripts/download_data.py")],
    capture_output=True, text=True)
print(r.stdout[-600:])
assert r.returncode == 0, r.stderr[-800:]

In [ ]:
raw = pd.read_csv(PROJECT / "data/raw/paysim.csv", usecols=["type", "isFraud"])
summary = raw.groupby("type").agg(transactions=("isFraud", "size"),
                                  frauds=("isFraud", "sum"))
print(f"total {len(raw):,} transactions / {raw.isFraud.sum():,} frauds")
summary

**Fraud occurs only in CASH_OUT and TRANSFER transactions.** If banks differ
in their transaction-type mix, they will differ enormously in how many fraud
labels they hold - and that is exactly the scenario this lab builds next.

---
## 3. Creating 10 Virtual Banks (non-IID partition)

After preprocessing (log1p on the money columns + StandardScaler, stratified
80/20 train/test split), the training set is distributed across 10 banks with
a **Dirichlet(alpha=0.5) split per transaction type**. Each bank ends up with
a different business profile, so fraud-label ownership varies by an order of
magnitude - a realistic setup. An IID partition (used by other experiments in
this repository) is generated as well (~2 minutes).

In [ ]:
for args in ([], ["--non-iid"]):
    r = subprocess.run([sys.executable,
                        str(PROJECT / "scripts/preprocess.py"), *args],
                       capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[-800:]
print("preprocess done")

from fl_aml.task import FEATURES, load_test
from sklearn.model_selection import train_test_split

# Split each bank's data once more into (solo-training / per-bank evaluation)
banks = []
for i in range(10):
    df = pd.read_parquet(PROJECT / f"data/partitions_noniid/part_{i}.parquet")
    tr, ev = train_test_split(df, test_size=0.2, stratify=df.isFraud,
                              random_state=42)
    banks.append({"train": tr, "eval": ev})

X_global, y_global = load_test(str(PROJECT / "data"))  # full-distribution test set

info = pd.DataFrame({
    "train_rows": [len(b["train"]) for b in banks],
    "train_frauds": [int(b["train"].isFraud.sum()) for b in banks],
    "eval_rows": [len(b["eval"]) for b in banks],
    "eval_frauds": [int(b["eval"].isFraud.sum()) for b in banks],
})
info.index.name = "bank"
print(f"global test set: {len(X_global):,} rows / {int(y_global.sum()):,} frauds")
info

Look at the `train_frauds` column. Some banks hold over 1,000 fraud examples,
while others (banks 3 and 6) hold only **a few dozen**. Can those banks build
a decent model alone? The next section finds out.

---
## 4. Before FL: Every Bank for Itself

Each bank trains a logistic regression **on its own data only**, to
convergence. We then cross-evaluate all 10 solo models against (a) every
bank's evaluation data and (b) the global test set that represents the full
transaction distribution.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score

solo_models = []
for i, b in enumerate(banks):
    m = LogisticRegression(max_iter=200)
    m.fit(b["train"][FEATURES].to_numpy(), b["train"].isFraud.to_numpy())
    solo_models.append(m)
    print(f"bank {i} solo model trained ({len(b['train']):>9,} rows, "
          f"{int(b['train'].isFraud.sum()):>5,} frauds)")

In [ ]:
# cross-evaluation: row = which bank's model, column = which bank's data (+ GLOBAL)
def eval_f1(model_predict, X, y):
    return f1_score(y, model_predict(X), zero_division=0)

col_names = [f"bank{j}" for j in range(10)] + ["GLOBAL"]
cross = np.zeros((10, 11))
for i, m in enumerate(solo_models):
    for j, b in enumerate(banks):
        cross[i, j] = eval_f1(m.predict, b["eval"][FEATURES].to_numpy(),
                              b["eval"].isFraud.to_numpy())
    cross[i, 10] = eval_f1(m.predict, X_global, y_global)

solo_global_f1 = cross[:, 10].copy()
print("done: 10 models x 11 datasets")

### 4.1 Heatmap: the solo-training world

Lighter cells mean worse (low F1); darker cells mean better. The number in
parentheses on each row is how many fraud labels that bank owns.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Style: a single-hue blue sequential ramp (light -> dark)
BLUES = LinearSegmentedColormap.from_list(
    "seq_blue", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
INK, MUTED = "#0b0b0b", "#898781"

def f1_heatmap(matrix, row_labels, col_labels, title, highlight_row=None):
    fig, ax = plt.subplots(
        figsize=(0.95 * len(col_labels) + 2.5, 0.52 * len(row_labels) + 1.6),
        dpi=110)
    im = ax.imshow(matrix, cmap=BLUES, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(col_labels)), col_labels, fontsize=9)
    ax.set_yticks(range(len(row_labels)), row_labels, fontsize=9)
    ax.set_xlabel("Evaluation data", fontsize=10, color=INK)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            v = matrix[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if v > 0.55 else INK)
    if highlight_row is not None:
        ax.add_patch(plt.Rectangle((-0.5, highlight_row - 0.5),
                                   len(col_labels), 1, fill=False,
                                   edgecolor="#eb6834", lw=2.2))
    ax.set_title(title, fontsize=12, pad=10)
    for s in ax.spines.values():
        s.set_visible(False)
    cb = fig.colorbar(im, ax=ax, shrink=0.8, label="F1 Score")
    cb.outline.set_visible(False)
    plt.tight_layout(); plt.show()

row_labels = [f"bank{i} solo (fraud {int(b['train'].isFraud.sum()):,})"
              for i, b in enumerate(banks)]
f1_heatmap(cross, row_labels, col_names,
           "Before FL: F1 of each bank's solo model (row = model, column = evaluation data)")

How to read it, and what to notice:

1. **Entire rows are pale (bad) for some banks** - a bank with only a few
   dozen or a few hundred fraud labels can barely detect fraud even on its
   own data (F1 around 0.1-0.3 in the GLOBAL column).
2. Even the fraud-rich banks top out around F1 ~0.65 on GLOBAL, because each
   solo model is over-fitted to its own transaction mix.
3. Regulation prevents the banks from simply pooling their data. This heatmap
   is the "before FL" reality.

Keep this picture in mind - in Section 8 we will redraw it with one extra row
for the federated model.

---
## 5. The Principle: FedAvg by Hand

One FL round is just three steps: (1) every bank starts from the global model
and trains locally, (2) each bank sends back **only its parameters** (no data
moves), and (3) the server forms the new global model as a data-size weighted
average: `w_global = sum(n_k * w_k) / sum(n_k)`.

Let's run two rounds by hand with just two banks: bank 6 (fewest frauds) and
bank 8 (most frauds).

In [ ]:
from fl_aml.task import (create_lr, get_lr_params, set_lr_params,
                         initial_lr_params, classification_metrics)

pair = {6: banks[6], 8: banks[8]}
global_params = initial_lr_params()          # coef = 0, intercept = 0

for round_no in (1, 2):
    local_params, local_sizes = [], []
    for bid, b in pair.items():
        model = create_lr(local_epochs=10, class_weight="none")
        set_lr_params(model, global_params)                # (1) start from global
        model.fit(b["train"][FEATURES].to_numpy(), b["train"].isFraud.to_numpy())
        m = classification_metrics(y_global, model.predict(X_global))
        print(f"round {round_no} bank{bid} local  : F1={m['f1']:.3f}")
        local_params.append(get_lr_params(model))          # (2) send parameters only
        local_sizes.append(len(b["train"]))
    total = sum(local_sizes)                               # (3) weighted average
    global_params = [sum(n * p[k] for n, p in zip(local_sizes, local_params)) / total
                     for k in range(2)]
    agg = create_lr(1, "none"); set_lr_params(agg, global_params)
    m = classification_metrics(y_global, agg.predict(X_global))
    print(f"round {round_no} AGGREGATED   : F1={m['f1']:.3f}\n")

print("Everything that crossed the 'network': 7 floats per bank "
      "(6 coefficients + 1 intercept). Zero transaction records.")

Bank 6, with only 18 fraud labels, ends up sharing a model it could never
build alone. Flower is the framework that performs this exact computation
over a real network (gRPC), for any number of nodes, with authentication and
TLS handled for you. The mapping from the hand computation to this
repository's code:

| Hand computation (above) | Flower code | File |
|---|---|---|
| `set_lr_params` then `fit` | `@app.train()` receives `msg.content["arrays"]`, then trains | `fl_aml/client_app.py` |
| send parameters only | `ArrayRecord` in the returned `Message` | `fl_aml/client_app.py` |
| weighted average | `FedAvg` strategy (`weighted_by_key="num-examples"`) | `fl_aml/server_app.py` |
| evaluate the aggregate | `strategy.start(..., evaluate_fn=...)` | `fl_aml/server_app.py` |

To inspect the client code from here:
`print((PROJECT / "fl_aml/client_app.py").read_text())`

---
## 6. Starting the Distributed Infrastructure: SuperLink + 10 SuperNodes

```
flwr run (submitted from this notebook in Section 7)
      |
      v :9093 (Exec API)
+-------------+   FedAvg aggregation + centralized evaluation
|  SuperLink  |
+-------------+
      ^ :9092 (Fleet API, gRPC - only parameters travel)
      |
 SN-0 ... SN-9   10 SuperNodes = 10 banks (ports 9094-9103)
```

> ### RUN IN TERMINAL
> SuperLink and the SuperNodes are long-lived background daemons, so they are
> started from a terminal (not from this notebook). Open a terminal on this
> machine and run:
>
> ```bash
> cd <repository-root>
> bash scripts/start_superlink.sh
> bash scripts/start_supernodes.sh 10
> ```

For a multi-machine deployment the only changes are pointing each SuperNode's
`--superlink 127.0.0.1:9092` at the server's address and replacing
`--insecure` with TLS certificates (see the README).

After running the terminal commands, verify from the notebook:

In [ ]:
def check_stack(min_nodes=10):
    r = subprocess.run(["bash", str(PROJECT / "scripts/status_check.sh")],
                       capture_output=True, text=True)
    alive = r.stdout.count("ALIVE  supernode")
    link = "ALIVE  superlink" in r.stdout
    assert link, "SuperLink is not running. Run the terminal commands above first."
    assert alive >= min_nodes, (
        f"Only {alive}/{min_nodes} SuperNodes are alive. "
        "Run: bash scripts/start_supernodes.sh 10")
    print(f"OK: SuperLink 1 + SuperNodes {alive}")

check_stack()

---
## 7. The FL Run: 10 Banks Train an MLP Together (10 rounds, ~10 minutes)

This time the model is an **MLP (6-32-16-1)**. It uses the same non-IID
partitions as the solo experiment in Section 4, and each bank's data still
never leaves its node. The cell below **submits the run asynchronously** and
returns immediately; the app bundle is shipped to the SuperLink, which
distributes it to every SuperNode.

In [ ]:
import shutil

RUN_DIR = PROJECT / "results/notebook_mlp_noniid"
shutil.rmtree(RUN_DIR, ignore_errors=True)     # clear any previous run

(PROJECT / "logs").mkdir(exist_ok=True)
run_log = open(PROJECT / "logs/notebook_run_mlp_noniid.log", "w")
proc = subprocess.Popen(
    ["bash", str(PROJECT / "scripts/run_experiment.sh"),
     "mlp", "10", "10", "partitions_noniid", "notebook_mlp_noniid"],
    stdout=run_log, stderr=subprocess.STDOUT, cwd=str(PROJECT))
print(f"submitted (pid {proc.pid}). Run the monitoring cell below.")

### 7.1 Monitoring the run

The server flushes its metrics to JSON at the end of every round, so polling
that file shows live progress. This cell waits until all 10 rounds finish
(~10 minutes).

In [ ]:
import time

agg_path = RUN_DIR / "aggregated_metrics.json"
seen = 0
while True:
    if proc.poll() is not None and proc.returncode != 0:
        raise RuntimeError("run failed - check logs/notebook_run_mlp_noniid.log")
    if agg_path.exists():
        agg = json.loads(agg_path.read_text())
        for e in agg[seen:]:
            print(f"  round {e['round']:>2}: P={e['precision']:.3f} "
                  f"R={e['recall']:.3f} F1={e['f1']:.3f}")
        seen = len(agg)
        if seen >= 11:                     # round 0 + 10 rounds
            break
    time.sleep(15)

while proc.poll() is None:
    time.sleep(5)
print("\nfederated training complete")

---
## 8. After FL: Before and After in a Single Picture

We restore the final global model the server saved (`final_model.pkl`) and
subject it to **exactly the same evaluation** as Section 4 (every bank's
evaluation data plus the global test set), then add it to the heatmap as one
extra row.

In [ ]:
import pickle
from fl_aml.task import MLP, predict_mlp

nds = pickle.loads((RUN_DIR / "final_model.pkl").read_bytes())
fl_model = MLP()
fl_model.load_state_dict({k: torch.tensor(a) for k, a
                          in zip(fl_model.state_dict().keys(), nds)})

fl_row = np.array(
    [eval_f1(lambda X: predict_mlp(fl_model, X),
             b["eval"][FEATURES].to_numpy(), b["eval"].isFraud.to_numpy())
     for b in banks]
    + [eval_f1(lambda X: predict_mlp(fl_model, X), X_global, y_global)])

full = np.vstack([cross, fl_row])
labels = row_labels + ["FL federated model (MLP)"]
f1_heatmap(full, labels, col_names,
           "Before vs After: solo models with the federated model added as the last row",
           highlight_row=10)

print(f"FL federated model GLOBAL F1 = {fl_row[-1]:.3f}  "
      f"(best solo: {solo_global_f1.max():.3f}, worst solo: {solo_global_f1.min():.3f})")

The orange-outlined **FL row is uniformly dark across every column.** In the
solo world only the fraud-rich banks had dark rows; in the federated world
the bank with 18 fraud labels runs the very same model. And on the GLOBAL
column, the federated model **beats even the best-endowed bank's solo
model** - collaboration pays off for the strongest participant too.

### 8.1 Per-bank Before vs After (GLOBAL test)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4), dpi=110)
xs = np.arange(10)
ax.bar(xs - 0.2, solo_global_f1, width=0.4, color="#eb6834", alpha=0.85,
       label="Before: solo model (LR)")
ax.bar(xs + 0.2, [fl_row[-1]] * 10, width=0.4, color="#2a78d6",
       label="After: FL federated model (MLP, shared by all banks)")
for x, v in zip(xs, solo_global_f1):
    ax.text(x - 0.2, v + 0.015, f"{v:.2f}", ha="center", fontsize=8, color=INK)
ax.text(9 + 0.2, fl_row[-1] + 0.015, f"{fl_row[-1]:.2f}", ha="center",
        fontsize=8, color=INK)
ax.set_xticks(xs, [f"bank{i}" for i in range(10)])
ax.set_ylabel("F1 on GLOBAL test"); ax.set_ylim(0, 1)
ax.grid(True, axis="y", color="#e1e0d9", lw=0.7); ax.set_axisbelow(True)
for s in ("top", "right"): ax.spines[s].set_visible(False)
ax.legend(frameon=False, fontsize=9, loc="upper left")
ax.set_title("Detection performance each bank gets on the full distribution",
             fontsize=12)
plt.tight_layout(); plt.show()

### 8.2 Confusion Matrices: a label-poor bank's day, before and after

The global test set contains 1,643 real frauds. Compare how many of them are
caught by bank 2's solo model (trained on just 185 fraud labels) versus the
federated model.

In [ ]:
from sklearn.metrics import confusion_matrix

def cm_plot(ax, y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    norm = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(norm, cmap=BLUES, vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:,}\n({norm[i, j]*100:.1f}%)",
                    ha="center", va="center", fontsize=10,
                    color="white" if norm[i, j] > 0.55 else INK)
    ax.set_xticks([0, 1], ["predicted normal", "predicted fraud"], fontsize=9)
    ax.set_yticks([0, 1], ["actual normal", "actual fraud"], fontsize=9)
    ax.set_title(title, fontsize=11)
    for s in ax.spines.values(): s.set_visible(False)

y_before = solo_models[2].predict(X_global)
y_after = predict_mlp(fl_model, X_global)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4), dpi=110)
cm_plot(axes[0], y_global, y_before,
        f"Before: bank2 solo (caught {int(((y_before==1)&(y_global==1)).sum()):,}"
        f"/{int(y_global.sum()):,} frauds)")
cm_plot(axes[1], y_global, y_after,
        f"After: FL federated (caught {int(((y_after==1)&(y_global==1)).sum()):,}"
        f"/{int(y_global.sum()):,} frauds)")
plt.tight_layout(); plt.show()

### 8.3 Round-by-Round Convergence: watching the collaboration form

Using the metrics saved during the Section 7 run, plot each bank's local
model (orange) and the aggregated model (blue) across rounds.

In [ ]:
agg = json.loads((RUN_DIR / "aggregated_metrics.json").read_text())
loc = pd.DataFrame(json.loads((RUN_DIR / "local_metrics.json").read_text()))
rounds = sorted(loc["round"].unique())
x = [r - 1 for r in rounds]

fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=110)
for ax, metric, title in zip(axes, ("precision", "f1"), ("Precision", "F1 Score")):
    for pid, g in loc.groupby("partition-id"):
        ax.plot(x, g.sort_values("round")[metric], color="#eb6834",
                lw=0.9, alpha=0.75)
    ax.plot(x, [next(e[metric] for e in agg if e["round"] == r) for r in rounds],
            color="#2a78d6", lw=2.4)
    ax.set_title(title); ax.set_xlabel("Round")
    ax.grid(True, color="#e1e0d9", lw=0.7); ax.set_axisbelow(True)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
axes[1].plot([], [], color="#eb6834", lw=0.9, label="Individual local model")
axes[1].plot([], [], color="#2a78d6", lw=2.4, label="Aggregated model")
axes[1].legend(loc="lower right", frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

---
## 9. Is the Comparison Fair? (optional)

A reasonable objection: "the solo models are logistic regression while FL
uses an MLP - isn't that the model's doing?" The cell below tests it: bank 0,
one of the best-endowed banks, trains the **same MLP architecture** alone to
convergence (60 epochs, ~3 minutes). Set the flag to True to run it.

In [ ]:
RUN_SOLO_MLP = False    # set to True to run (~3 minutes)

if RUN_SOLO_MLP:
    from fl_aml.task import train_mlp
    torch.manual_seed(0)
    solo_mlp = MLP()
    b = banks[0]
    train_mlp(solo_mlp, b["train"][FEATURES].to_numpy(),
              b["train"].isFraud.to_numpy(), epochs=60)
    f1_solo = eval_f1(lambda X: predict_mlp(solo_mlp, X), X_global, y_global)
    print(f"bank0 solo MLP (60 epochs)   GLOBAL F1 = {f1_solo:.3f}")
    print(f"FL federated MLP (10 rounds) GLOBAL F1 = {fl_row[-1]:.3f}")
    print("Same architecture - but one bank's data alone cannot reach the "
          "federated model.")
else:
    print("Set RUN_SOLO_MLP = True to compare a solo MLP with the same architecture.")

Related experiments in this repository (IID replication with logistic
regression, an IID MLP run, and more) can be reproduced on the same
infrastructure, e.g. `bash scripts/run_experiment.sh logreg 10 10`. See the
README for details.

---
## 10. Teardown and Next Steps

> ### RUN IN TERMINAL
> When you are done, stop the FL processes from a terminal:
>
> ```bash
> cd <repository-root>
> bash scripts/stop_all.sh
> ```

What this notebook demonstrated:

1. **Before**: solo-training quality is dictated by how many fraud labels a
   bank owns; label-poor banks were effectively blind (the pale heatmap rows).
2. **The principle**: FedAvg is just a weighted average of parameters - not a
   single raw transaction ever moves.
3. **After**: the federated model was uniformly strong for every bank, and on
   the global benchmark it surpassed even the best-endowed bank's solo model.

Next steps: multi-machine deployment (see the README), TLS + node
authentication, differential-privacy mods, gradient-boosted models (FedXGB),
and threshold tuning for operational precision/recall trade-offs.